In [ ]:
# system libs
import os, sys, glob
import datetime
import xarray as xr    
import matplotlib.pyplot as plt
# array operators and netcdf datasets
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)
import pandas as pd
# to have tools to format time
sys.path.append( '/work/bb1224/2024_MS-COURSE/tools/analysis' )
from tools import convert_timevec

import warnings
warnings.simplefilter("ignore")


## re-load data as nested dicts

 re-load data as nested dicts: 
- Mask_fields[case][dom]
- Mask_fields_filtered[case][dom]
- Features_fields[case][dom]
- Features_fields_filtered[case][dom]

In [ ]:
#re-load data as nested dicts
def load_fields(base_mask_path="output_masks", base_feat_path="output_features"):
    case_names = ["case1", "case2"]
    dom_names = ["DOM01", "DOM02", "DOM03"]

    Mask_fields = {}
    Mask_fields_filtered = {}
    Features_fields = {}
    Features_fields_filtered = {}

    for case in case_names:
        Mask_fields[case] = {}
        Mask_fields_filtered[case] = {}
        Features_fields[case] = {}
        Features_fields_filtered[case] = {}

        for dom in dom_names:
            mask_file = os.path.join(base_mask_path, f"mask_{case}_{dom}.nc")
            mask_file_filt = os.path.join(base_mask_path, f"mask_filt_{case}_{dom}.nc")
            feat_file = os.path.join(base_feat_path, f"features_{case}_{dom}.csv")
            feat_file_filt = os.path.join(base_feat_path, f"features_filt_{case}_{dom}.csv")
                        
            if os.path.exists(mask_file):
                Mask_fields[case][dom] = xr.load_dataarray(mask_file)

            if os.path.exists(mask_file):
                Mask_fields_filtered[case][dom] = xr.load_dataarray(mask_file_filt)

            if os.path.exists(feat_file):
                Features_fields[case][dom] = pd.read_csv(feat_file)
                
            if os.path.exists(feat_file):
                Features_fields_filtered[case][dom] = pd.read_csv(feat_file_filt)

    return Features_fields, Mask_fields, Mask_fields_filtered, Features_fields_filtered


In [ ]:
Features_fields, Mask_fields, Mask_fields_filtered, Features_fields_filtered = load_fields()


In [ ]:
Features_fields["case1"]["DOM01"] 

In [ ]:
Mask_fields["case1"]["DOM01"]

## PDF

In [ ]:
sizes = Features_fields["case2"]["DOM01"]["size_km2"].to_numpy()

#Define log-spaced bins 
nbins = 20
bin_edges = np.logspace(0, np.log10(sizes.max()), nbins+1) 

#Histogram + Normalize 
counts, bin_edges = np.histogram(sizes, bins=bin_edges) 
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:]) 
# geometric mean 
bin_widths = np.diff(bin_edges) 
pdf = counts / (counts.sum() * bin_widths) 

#Plot 
plt.figure() 
plt.loglog(bin_centers, pdf, marker='o',label='data') 
plt.xlabel("Cloud area $A$ [km$^2$]") 
plt.ylabel("PDF $p(A)$") 
plt.title("Log-binned PDF of Cloud Areas") 
plt.grid(True) 

#plot reference 
beta = 1.33
A0 = np.median(bin_centers)
p0 = 0.001 # pick by try and error
ref = p0 * (bin_centers / A0)**(-beta)
plt.loglog(bin_centers, ref, '--', label=r'ref: $A^{-1.33}$')
plt.legend()


plt.show()

In [ ]:
 def compute_pdf_logbins(sizes, nbins=20):
    """Compute log-binned PDF for a given 1D array of cloud sizes (km²)."""

    # remove zeros and extremely small values
    sizes = sizes[sizes > 0]

    # avoid too-small bins
    if sizes.min() <= 0:
        raise ValueError("sizes contain non-positive values")

    # log-spaced bin edges
    #bin_edges = np.logspace(0, np.log10(sizes.max()), nbins+1)
    bin_edges = np.logspace(np.log10(sizes.min()), np.log10(sizes.max()), nbins+1)

    counts, _ = np.histogram(sizes, bins=bin_edges)
    bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])
    bin_widths  = np.diff(bin_edges)

    pdf = counts / (counts.sum() * bin_widths)

    return bin_centers, pdf

def extract_sizes(features, case, dom):

    sizes = features[case][dom]["size_km2"].to_numpy()
    
    return sizes[sizes > 0]

In [ ]:
sizes_dom1 = extract_sizes(Features_fields, "case2", "DOM01")
sizes_dom2 = extract_sizes(Features_fields, "case2", "DOM02")
sizes_dom3 = extract_sizes(Features_fields, "case2", "DOM03")

bins = 20
bc1, pdf1 = compute_pdf_logbins(sizes_dom1, nbins=bins)
bc2, pdf2 = compute_pdf_logbins(sizes_dom2, nbins=bins)
bc3, pdf3 = compute_pdf_logbins(sizes_dom3, nbins=bins)


plt.figure(figsize=(8,6))

plt.loglog(bc1, pdf1, marker='o', label="DOM01")
plt.loglog(bc2, pdf2, marker='s', label="DOM02")
plt.loglog(bc3, pdf3, marker='^', label="DOM03")

plt.xlabel("Cloud area $A$ [km$^2$]")
plt.ylabel("PDF $p(A)$")
plt.title("Cloud Size PDFs — Case 2")
plt.legend()
plt.grid(True)


plt.legend()

plt.show()

In [ ]:
case = "case1"

fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True, constrained_layout=True)

# ---- UNFILTERED ----
sizes_dom1 = extract_sizes(Features_fields, case, "DOM01")
sizes_dom2 = extract_sizes(Features_fields, case, "DOM02")
sizes_dom3 = extract_sizes(Features_fields, case, "DOM03")

bc1, pdf1 = compute_pdf_logbins(sizes_dom1, nbins=15)
bc2, pdf2 = compute_pdf_logbins(sizes_dom2, nbins=15)
bc3, pdf3 = compute_pdf_logbins(sizes_dom3, nbins=15)

axs[0].loglog(bc1, pdf1, marker='o', label="DOM01")
axs[0].loglog(bc2, pdf2, marker='s', label="DOM02")
axs[0].loglog(bc3, pdf3, marker='^', label="DOM03")

axs[0].set_xlabel("Cloud area $A$ [km$^2$]")
axs[0].set_ylabel("PDF $p(A)$")
axs[0].set_title("Unfiltered Features — Case 2")
axs[0].grid(True)
axs[0].legend()

# ---- FILTERED ----
sizes_dom1_filt = extract_sizes(Features_fields_filtered, case, "DOM01")
sizes_dom2_filt = extract_sizes(Features_fields_filtered, case, "DOM02")
sizes_dom3_filt = extract_sizes(Features_fields_filtered, case, "DOM03")

bc1f, pdf1f = compute_pdf_logbins(sizes_dom1_filt, nbins=15)
bc2f, pdf2f = compute_pdf_logbins(sizes_dom2_filt, nbins=15)
bc3f, pdf3f = compute_pdf_logbins(sizes_dom3_filt, nbins=15)

axs[1].loglog(bc1f, pdf1f, marker='o', label="DOM01")
axs[1].loglog(bc2f, pdf2f, marker='s', label="DOM02")
axs[1].loglog(bc3f, pdf3f, marker='^', label="DOM03")

axs[1].set_xlabel("Cloud area $A$ [km$^2$]")
axs[1].set_title("Filtered Features — Case 2")
axs[1].grid(True)
axs[1].legend()

plt.suptitle("Cloud Size PDFs: Unfiltered vs. Filtered — Case 2", fontsize=14, fontweight="bold")
plt.show()


In [ ]:
plt.figure(figsize=(10, 7))

# --- Unfiltered ---
sizes_dom1 = extract_sizes(Features_fields, "case2", "DOM01")
sizes_dom2 = extract_sizes(Features_fields, "case2", "DOM02")
sizes_dom3 = extract_sizes(Features_fields, "case2", "DOM03")

bc1, pdf1 = compute_pdf_logbins(sizes_dom1, nbins=15)
bc2, pdf2 = compute_pdf_logbins(sizes_dom2, nbins=15)
bc3, pdf3 = compute_pdf_logbins(sizes_dom3, nbins=15)

plt.loglog(bc1, pdf1, 'o-', label="DOM01 (unfiltered)")
plt.loglog(bc2, pdf2, 's-', label="DOM02 (unfiltered)")
plt.loglog(bc3, pdf3, '^-', label="DOM03 (unfiltered)")

# --- Filtered ---
sizes_dom1_filt = extract_sizes(Features_fields_filtered, "case2", "DOM01")
sizes_dom2_filt = extract_sizes(Features_fields_filtered, "case2", "DOM02")
sizes_dom3_filt = extract_sizes(Features_fields_filtered, "case2", "DOM03")

bc1f, pdf1f = compute_pdf_logbins(sizes_dom1_filt, nbins=15)
bc2f, pdf2f = compute_pdf_logbins(sizes_dom2_filt, nbins=15)
bc3f, pdf3f = compute_pdf_logbins(sizes_dom3_filt, nbins=15)

plt.loglog(bc1f, pdf1f, 'o--', label="DOM01 (filtered)")
plt.loglog(bc2f, pdf2f, 's--', label="DOM02 (filtered)")
plt.loglog(bc3f, pdf3f, '^--', label="DOM03 (filtered)")

# Labels & legend
plt.xlabel("Cloud area $A$ [km$^2$]")
plt.ylabel("PDF $p(A)$")
plt.title("Cloud Size PDFs — Case 2")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
"""
Plot log-binned PDFs of cloud areas for one case across all domains.
Compares original vs boundary-filtered segmentation results.
"""

case_name = "case2"
domains = Features_fields[case_name].keys()

# 1. Extract all area values from both original and filtered
all_areas_unfiltered = []
all_areas_filtered = []

for dom in domains:
    df_orig = Features_fields[case_name][dom]
    all_areas_unfiltered.extend(df_orig["size_km2"].values)

    if case_name in Features_fields_filtered and dom in Features_fields_filtered[case_name]:
        df_filt = Features_fields_filtered[case_name][dom]
        all_areas_filtered.extend(df_filt["size_km2"].values)

all_areas_unfiltered = np.asarray(all_areas_unfiltered)
all_areas_filtered   = np.asarray(all_areas_filtered)



bc_unfilt, pdf_unfilt = compute_pdf_logbins(all_areas_unfiltered, nbins=15)
bc_filt, pdf_filt = compute_pdf_logbins(all_areas_filtered, nbins=15)



# 2. Plotting
plt.figure(figsize=(8,6))
plt.loglog(bc_unfilt, pdf_unfilt, marker='o', label="Unfiltered")
plt.loglog(bc_filt, pdf_filt, marker='s', label="Filtered")

plt.xlabel("Cloud area $A$ [km$^2$]")
plt.ylabel("PDF $p(A)$")
plt.title("Cloud Size PDFs — Case 1")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
fig.suptitle("Log-binned Cloud Area PDFs across all Domains", fontsize=15, fontweight='bold')

for i, case_name in enumerate(["case1", "case2"]):
    domains = Features_fields[case_name].keys()

    # 1. Collect areas
    all_areas_unfiltered = []
    all_areas_filtered   = []

    for dom in domains:
        df_orig = Features_fields[case_name][dom]
        all_areas_unfiltered.extend(df_orig["size_km2"].values)

        if case_name in Features_fields_filtered and dom in Features_fields_filtered[case_name]:
            df_filt = Features_fields_filtered[case_name][dom]
            all_areas_filtered.extend(df_filt["size_km2"].values)

    all_areas_unfiltered = np.asarray(all_areas_unfiltered)
    all_areas_filtered   = np.asarray(all_areas_filtered)
    print(len(all_areas_unfiltered))
    print(len(all_areas_filtered))
    #filter out small an big clouds
    all_areas_unfiltered = all_areas_unfiltered[all_areas_unfiltered > 5]
    all_areas_filtered   = all_areas_filtered[all_areas_filtered > 5]
    #all_areas_unfiltered = all_areas_unfiltered[all_areas_unfiltered < 150]
    #all_areas_filtered   = all_areas_filtered[all_areas_filtered < 150]


    # 2. Compute PDFs
    bc_unfilt, pdf_unfilt = compute_pdf_logbins(all_areas_unfiltered, nbins=15)
    bc_filt, pdf_filt     = compute_pdf_logbins(all_areas_filtered, nbins=15)

    # 3. Plot
    ax = axes[i]
    ax.loglog(bc_unfilt, pdf_unfilt, marker='o', label="Unfiltered")
    ax.loglog(bc_filt, pdf_filt, marker='s', label="Filtered")
    ax.set_title(f"{case_name}")
    ax.set_xlabel("Cloud area $A$ [km$^2$]")
    if i == 0:
        ax.set_ylabel("PDF $p(A)$")
    ax.grid(True)
    
    #plot reference 
    #bin_centers = bc_filt
    #beta = 1.33
    #A0 = np.median(bin_centers)
    #p0 = 0.02 # pick by try and error
    #ref = p0 * (bin_centers / A0)**(-beta)
    #ax.loglog(bin_centers, ref, '--', label=r'ref: $A^{-1.33}$')

    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
all_areas_unfiltered = []
all_areas_filtered   = []

for i, case_name in enumerate(["case1", "case2"]):
    domains = Features_fields[case_name].keys()

    # 1. Collect areas

    for dom in domains:
        df_orig = Features_fields[case_name][dom]
        all_areas_unfiltered.extend(df_orig["size_km2"].values)

        if case_name in Features_fields_filtered and dom in Features_fields_filtered[case_name]:
            df_filt = Features_fields_filtered[case_name][dom]
            all_areas_filtered.extend(df_filt["size_km2"].values)

all_areas_unfiltered = np.asarray(all_areas_unfiltered)
all_areas_filtered   = np.asarray(all_areas_filtered)


#filter out small an big clouds
all_areas_unfiltered = all_areas_unfiltered[(all_areas_unfiltered > 10) & (all_areas_unfiltered < 150)]
all_areas_filtered   = all_areas_filtered[(all_areas_filtered > 10) & (all_areas_filtered < 150)]

# 2. Compute PDFs
bc_unfilt, pdf_unfilt = compute_pdf_logbins(all_areas_unfiltered, nbins=15)
bc_filt, pdf_filt     = compute_pdf_logbins(all_areas_filtered, nbins=15)

# 3. Plot
plt.figure(figsize=(8,6))
plt.loglog(bc_unfilt, pdf_unfilt, marker='o', label="Unfiltered")
plt.loglog(bc_filt, pdf_filt, marker='s', label="Filtered")

plt.xlabel("Cloud area $A$ [km$^2$]")
plt.ylabel("PDF $p(A)$")
plt.title("Cloud Size PDFs across all Cases and Domains")
plt.legend()
plt.grid(True)

#plot reference 
bin_centers = bc_filt
beta = 1.33
A0 = np.median(bin_centers)
p0 = 0.01 # pick by try and error
ref = p0 * (bin_centers / A0)**(-beta)
plt.loglog(bin_centers, ref, '--', label=r'ref: $A^{-1.33}$')
plt.legend()

plt.tight_layout()
plt.show()


# PCF


In [ ]:
df = Features_fields["case1"]["DOM01"]
coords = df[["hdim_1", "hdim_2"]].to_numpy()

print(coords)


In [ ]:
Features_fields["case1"]["DOM01"]

In [ ]:
df = Features_fields["case1"]["DOM01"]
print(df["time"].unique())

In [ ]:
df = Features_fields["case1"]["DOM01"]

# Use a matching timestamp string
df_at_time = df[df["time"] == "2024-08-06 12:00:00"]

print(f"{len(df_at_time)} features found at 2024-08-06 12:00:00")
display(df_at_time)

In [ ]:
from scipy.spatial.distance import pdist, squareform

r_bins = np.linspace(0, 230, 21)  # 0 to 100 px in 20 bins
# 3. Compute bin centers
r_centers = (r_bins[:-1] + r_bins[1:]) / 2

# Filter by timestamp
df = Features_fields["case1"]["DOM01"]
df_at_time = df_at_time = df[df["time"] == "2024-08-06 12:00:00"]
##print(f"df_at_time:", df_at_time)

# Extract coordinates/positions at this time
positions = df_at_time[["hdim_1", "hdim_2"]].to_numpy()
print(f"positions:", positions)
print(f"len(positions):", len(positions))
print()

mask = Mask_fields["case1"]["DOM01"]
domain_shape = (mask.sizes["lon"], mask.sizes["lat"])
print(f"domain_shape:", domain_shape)
print()

n_points = positions.shape[0]
print(f"positions.shape[0]:",positions.shape[0])
print()
if n_points < 2:
    print(f"less than 2 objects", r_bins[:-1] + np.diff(r_bins)/2, np.zeros_like(r_bins[:-1]))


# 1. Compute pairwise distances
dists = pdist(positions)
print(f"dists",dists)
print(f"len(dists):",len(dists))
print()
    
# 2. Histogram distances
counts, _ = np.histogram(dists, bins=r_bins)
print(f"r_bins", r_bins)
print(f"r_centers:",r_centers)
print(f"counts",counts)
print()


In [ ]:
# 4. Plot the pairwise distance histogram

plt.figure(figsize=(7, 5))
plt.plot(r_centers, counts, marker='o')
plt.xlabel("Distance r [pixels]")
plt.ylabel("Number of pairs")
plt.title("Pairwise Distance Histogram")
plt.grid(True)
plt.show()

In [ ]:
#averaged over all timestamps

# Parameters
r_bins = np.linspace(0, 200, 21)  # distance bins (in pixels)
df = Features_fields["case2"]["DOM01"]
mask = Mask_fields["case2"]["DOM01"]
domain_shape = (mask.sizes["lon"], mask.sizes["lat"])

# Accumulate histogram counts
total_counts = np.zeros(len(r_bins) - 1)
valid_snapshots = 0

# Loop through all unique timestamps in the features catalog
for timestamp in df["time"].unique():
    df_at_time = df[df["time"] == timestamp]
    positions = df_at_time[["hdim_1", "hdim_2"]].to_numpy()

    if positions.shape[0] < 2:
        continue  # skip if fewer than 2 objects

    dists = pdist(positions)
    counts, _ = np.histogram(dists, bins=r_bins)

    total_counts += counts
    valid_snapshots += 1

# Normalize
avg_counts = total_counts / valid_snapshots if valid_snapshots > 0 else total_counts
r_centers = (r_bins[:-1] + r_bins[1:]) / 2

# Plot
plt.figure(figsize=(7, 5))
plt.plot(r_centers, avg_counts, marker='o')
plt.xlabel("Distance $r$ [pixels]")
plt.ylabel("Average number of pairs")
plt.title("Mean Pairwise Distance Histogram — Averaged over all snapshots")
plt.grid(True)
plt.show()


In [ ]:
#reference distribution


x_min, x_max = 0, domain_shape[0]   
y_min, y_max = 0, domain_shape[1]

#N_obj= np.int(sum(avg_counts))
N_obj= 1000  # number of sampled objects


# Generate N_obj random point coordinates uniformly in the domain
x_points = np.random.uniform(x_min, x_max, N_obj)
y_points = np.random.uniform(y_min, y_max, N_obj)

# Combine x and y coordinates into an array of shape (N_obs, 2)
random_positions = np.stack([x_points, y_points], axis=1)

# Compute pairwise distances
dists_random = pdist(random_positions)

# Use the same bins as the observed data
counts_random, _ = np.histogram(dists_random, bins=r_bins)

# Normalize (optional, to compare shapes rather than raw counts)
counts_random = counts_random / counts_random.sum()
counts_obs = avg_counts / avg_counts.sum()

# Plot comparison
plt.figure(figsize=(8,5))
plt.plot(r_centers, counts_obs, label='Observed', marker='o')
plt.plot(r_centers, counts_random, label='Reference (Random)', marker='o')
plt.xlabel("Pairwise distance [pixels]")
plt.ylabel("Normalized count")
plt.title("Observed vs Random Reference Pair Distance Distribution")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Plot g(r)
plt.figure(figsize=(8,5))
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
plt.plot(r_centers, counts_obs/counts_random, label='g(r)', marker='o')
plt.xlabel("Pairwise distance [pixels]")
plt.ylabel("g(r)")
plt.title("PCF")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
def compute_random_reference(domain_shape, N_obj, r_bins):
    """Simulate random points and compute reference pairwise distance histogram."""
    x_min, x_max = 0, domain_shape[0]   
    y_min, y_max = 0, domain_shape[1]

    x_points = np.random.uniform(x_min, x_max, N_obj)
    y_points = np.random.uniform(y_min, y_max, N_obj)
    random_positions = np.stack([x_points, y_points], axis=1)

    dists_random = pdist(random_positions)
    counts_random, _ = np.histogram(dists_random, bins=r_bins)
    return counts_random


In [ ]:

def compute_avg_pairwise_counts(df, r_bins):
    """Compute average pairwise distance histogram across all timestamps."""
    total_counts = np.zeros(len(r_bins) - 1)
    valid_snapshots = 0

    for timestamp in df["time"].unique():
        df_at_time = df[df["time"] == timestamp]
        positions = df_at_time[["hdim_1", "hdim_2"]].to_numpy()
        if positions.shape[0] < 2:
            continue
        dists = pdist(positions)
        counts, _ = np.histogram(dists, bins=r_bins)
        total_counts += counts
        valid_snapshots += 1

    avg_counts = total_counts / valid_snapshots if valid_snapshots > 0 else total_counts
    return avg_counts




In [ ]:
def compute_avg_pairwise_counts_all_domains(case_name, Features_fields, Mask_fields, r_bins):
    total_counts = np.zeros(len(r_bins) - 1)
    valid_snapshots = 0

    for dom in Features_fields[case_name]:
        df = Features_fields[case_name][dom]
        for timestamp in df["time"].unique():
            df_at_time = df[df["time"] == timestamp]
            positions = df_at_time[["hdim_1", "hdim_2"]].to_numpy()
            if positions.shape[0] < 2:
                continue
            dists = pdist(positions)
            counts, _ = np.histogram(dists, bins=r_bins)
            total_counts += counts
            valid_snapshots += 1

    avg_counts = total_counts / valid_snapshots if valid_snapshots > 0 else total_counts
    return avg_counts

In [ ]:
case_name = "case2"
r_bins = np.linspace(0, 200, 21)


df = Features_fields[case_name][dom]
mask = Mask_fields[case_name][dom]
domain_shape = (mask.sizes["lon"], mask.sizes["lat"])

# Compute average counts and random reference
#counts_obs = compute_avg_pairwise_counts(df, r_bins)
counts_obs = compute_avg_pairwise_counts_all_domains(case_name, Features_fields, Mask_fields, r_bins)

counts_random = compute_random_reference(domain_shape, N_obj, r_bins)

# Normalize for PCF
norm_obs = counts_obs / counts_obs.sum() if counts_obs.sum() > 0 else counts_obs
norm_rand = counts_random / counts_random.sum() if counts_random.sum() > 0 else counts_random

# Plot comparison
plt.figure(figsize=(8,5))
plt.plot(r_centers, norm_obs, label='Observed', marker='o')
plt.plot(r_centers, norm_rand, label='Reference (Random)', marker='o')
plt.xlabel("Pairwise distance [pixels]")
plt.ylabel("Normalized count")
plt.title("Observed vs Random Reference Pair Distance Distribution")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(r_centers, norm_obs/norm_rand, label='Observed', marker='o')

plt.xlabel("Pairwise distance [pixels]")
plt.ylabel("Normalized count")
plt.title("Observed vs Random Reference Pair Distance Distribution")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
r_bins = np.linspace(0, 200, 21)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
cases = ["case1", "case2"]
N_obj = 1000  # number of random points for reference

for i, case_name in enumerate(cases):
    # Compute observed counts averaged over all domains
    counts_obs = compute_avg_pairwise_counts_all_domains(case_name, Features_fields, Mask_fields, r_bins)

    # Use one domain’s shape for reference generation
    example_dom = next(iter(Mask_fields[case_name]))
    mask = Mask_fields[case_name][example_dom]
    domain_shape = (mask.sizes["lon"], mask.sizes["lat"])
    counts_rand = compute_random_reference(domain_shape, N_obj, r_bins)

    # Normalize
    norm_obs = counts_obs / counts_obs.sum() if counts_obs.sum() > 0 else counts_obs
    norm_rand = counts_rand / counts_rand.sum() if counts_rand.sum() > 0 else counts_rand
    g_r = norm_obs / norm_rand

    # --- Left plot: counts ---
    axs[i, 0].plot(r_centers, norm_obs, label='Observed', marker='o')
    axs[i, 0].plot(r_centers, norm_rand, label='Random', marker='o')
    axs[i, 0].set_ylabel(f"{case_name} — Normalized count")
    axs[i, 0].legend()
    axs[i, 0].grid(True)

    # --- Right plot: PCF ---
    axs[i, 1].plot(r_centers, g_r, marker='o', label='g(r)')
    axs[i, 1].axhline(1, color='red', linestyle='--', linewidth=1)
    axs[i, 1].set_ylabel(f"{case_name} — g(r)")
    axs[i, 1].legend()
    axs[i, 1].grid(True)

# Common labels
for ax in axs[-1]:
    ax.set_xlabel("Distance $r$ [pixels]")

axs[0, 0].set_title("Observed vs Random Reference")
axs[0, 1].set_title("Pair Correlation Function (PCF)")

plt.tight_layout()
plt.show()


## Organization index based on Cloud Cover

- Convective Organization Potential (COP): assuming that larger and closer objects are more likely 
to interact with each other. Based on the concept of gravitational interaction potential, it is
determined from the distance between centers of objects and the radii of equal area circles.
Higher COP values indicate higher degrees on organization. Doesn’t need a comparison to a
random distribution and considers both object size and spacing.

- Morphological Index of Convetive Aggregation (MICA): Considers besides the density of objects in a
confined area also the amount of space on the domain where no convection occours (fraction of
cloud and clear sky coverage)

- Organization Index based on Distance and Relative Area (OIDRA): combines the advantages of previous indicies
while minimizing their drawbacks. It considers both the distances between object edges and their
sizes. OIDRA explicitly does not depend on the size of the objects but only on their relative
fraction.


### Convective Organization Potential(COP):

White et al. (2018):

Euqal-area radius: $ r_i = \left( \frac{A_i}{\pi} \right)^{1/2} $

Pair interaction potential: $ V(i,j) = \frac{r_i + r_j}{d_{ij}}$

Same expression written directly in terms of areas: $ V(i,j) = \frac{\sqrt{A_i / \pi} + \sqrt{A_j / \pi}}{d_{ij}} $

Convective Organization Potential (COP): 
Mean over all unique pairs: $ \mathrm{COP} = \frac{2}{N(N-1)} \sum_{i<j} V(i,j) $

Equivalent double-sum form: $ \mathrm{COP} = \frac{1}{\tfrac{1}{2}N(N-1)} 
\sum_{i=1}^{N} \sum_{j=i+1}^{N} 
\frac{r_i + r_j}{d_{ij}}
$



A higher COP value indicates that large convective cells are packed close together (tightly clustered), which corresponds to a greater degree of organization.


In [ ]:
Features_fields_filtered["case1"]["DOM01"] 

## COP implementation from scratch

In [ ]:
df = Features_fields_filtered["case2"]["DOM01"] 

df_t = df[df["frame"] == 0] #df_t contains only one frame

xy = df_t[["hdim_1", "hdim_2"]].to_numpy(dtype=float)   # shape (N, 2)
A  = df_t["ncells"].to_numpy(dtype=float)               # shape (N,)
N  = len(A)


In [ ]:
print(xy.shape, A.shape, N)
print("min/max area:", A.min(), A.max())

In [ ]:
#Convert areas to radii 
r = np.sqrt(A / np.pi)   # radii in pixel units
print("min/max r:", r.min(), r.max())


In [ ]:
# Compute all pair distances with pdist

from scipy.spatial.distance import pdist

d = pdist(xy, metric="euclidean")   # shape (M,)

In [ ]:
M = N * (N - 1) // 2
print("len(d) =", len(d), "expected M =", M)
print("min distance:", d.min(), "max distance:", d.max())

In [ ]:
# building the mathcing pairwise (ri +rj) vector

i, j = np.triu_indices(N, k=1)
rsum = r[i] + r[j]     # shape (M,)

In [ ]:
print("len(rsum) =", len(rsum), "should equal len(d) =", len(d))

In [ ]:
# compute pair inteaction potentials V

eps = 1e-12 # avoid devision by zero, use a tiny eps
V = rsum / np.maximum(d, eps)


In [ ]:
print("V min/max:", V.min(), V.max())

In [ ]:
#average over all pairs to get COP

COP = V.mean()   # because mean = sum / M

In [ ]:
print("COP =", COP)

In [ ]:

def cop_one_frame(df_t, eps=1e-12):
    """
    COP (White et al. style) for a single scene/frame.

    Uses:
      - centroid coords: hdim_1, hdim_2
      - area: ncells  (pixels)
      - pair distances: pdist

    Returns:
      float COP, or np.nan if N < 2
    """
    N = len(df_t)
    if N < 2:
        return np.nan
        
    # centroids in pixel space
    xy = df_t[["hdim_1", "hdim_2"]].to_numpy(dtype=float) 

    # areas in pixels -> radii in pixels
    A  = df_t["ncells"].to_numpy(dtype=float)
    r = np.sqrt(A / np.pi)

    # pairwise distances (condensed form, length = N*(N-1)/2)
    d = pdist(xy, metric="euclidean")
    d = np.maximum(d, eps)

    # pairwise (r_i + r_j) in the same condensed order as pdist
    i, j = np.triu_indices(N, k=1) #np.triu_indices(N, k=1) gives the right (i,j) pairs order
    rsum = r[i] + r[j]

    V = rsum / d
    return V.mean()


In [ ]:
#Sanity check
df = Features_fields_filtered["case1"]["DOM01"]

df_t2 = df_t.copy()
df_t2["hdim_1"] *= 2
df_t2["hdim_2"] *= 2
print("COP original:", cop_one_frame(df_t))
print("COP stretched:", cop_one_frame(df_t2))


### Clean functions

In [ ]:
#turn it into a time series (groupby frame)

def cop_timeseries(df, group_key="frame"):
    """
    Compute COP for each frame in a feature dataframe.
    Returns a DataFrame with one row per frame.
    """
    rows = []
    for key, df_t in df.groupby(group_key, sort=True):
        rows.append({
            group_key: key,
            "time": df_t["time"].iloc[0],
            "N_objects": len(df_t),
            "COP": cop_one_frame(df_t),
        })
    return pd.DataFrame(rows).sort_values(group_key).reset_index(drop=True)


In [ ]:
# usage 

df = Features_fields_filtered["case1"]["DOM01"]
cop_df1 = cop_timeseries(df, group_key="frame")


In [ ]:
cop_df1.head()


In [ ]:
plt.figure(figsize=(10,5))

plt.plot(cop_df1["frame"], cop_df1["COP"], lw=1.8)

plt.xlabel("Time")
plt.ylabel("COP")
plt.title(f"COP time series – {case} – {dom}")
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
cases = ["case1", "case2"]
domains = ["DOM01", "DOM02", "DOM03"]

cop_all = {}

for case in cases:
    cop_all[case] = {}
    for dom in domains:
        df = Features_fields_filtered[case][dom]
        cop_df = cop_timeseries(df)
        cop_all[case][dom] = cop_df[["time", "COP"]].rename(
            columns={"COP": f"COP_{dom}"}
        )



In [ ]:
cop_compare = {}

for case in cases:
    df_merge = cop_all[case]["DOM01"]

    for dom in ["DOM02", "DOM03"]:
        df_merge = df_merge.merge(
            cop_all[case][dom],
            on="time",
            how="inner"
        )

    cop_compare[case] = df_merge


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5), sharey=True)

for ax, case in zip(axes, cases):
    df_plot = cop_compare[case]

    ax.plot(df_plot["time"], df_plot["COP_DOM01"], label="DOM01", lw=2)
    ax.plot(df_plot["time"], df_plot["COP_DOM02"], label="DOM02", lw=2)
    ax.plot(df_plot["time"], df_plot["COP_DOM03"], label="DOM03", lw=2)

    ax.set_title(case)
    ax.set_xlabel("Time")
    ax.grid(alpha=0.3)


axes[0].set_ylabel("COP")
axes[1].legend(loc="upper right")

fig.suptitle("COP comparison across domains", fontsize=14)

#fixing spacing & rotation of date/time
fig.autofmt_xdate()   
for ax in axes:
    ax.set_xticks(ax.get_xticks()[::3])  # keep every 3rd tick


plt.tight_layout()
plt.show()
